In [13]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold
from tqdm import tqdm

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.myutils import *
from drGAT.sampler import RandomSampler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg, _, _, _ = load_data("nci")

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:03<00:00,  7.64it/s]


Done!


In [10]:
PATH = "../nci_data/"

In [14]:
k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

for seed, (train_index, test_index) in enumerate(tqdm(kfold.split(np.arange(pos_num)))):
    sampler = RandomSampler(
        drugAct.T,
        train_index,
        test_index,
        null_mask.T,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed=seed,
    )
    break

0it [00:00, ?it/s]


In [15]:
sampler.train_data

tensor([[1., 1., 1.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 1., 0., 1.],
        [0., 0., 0.,  ..., 0., 1., 1.]])

In [16]:
class Args:
    def __init__(self):
        self.device = "cuda:0"  # cuda:number or cpu
        self.data = "nci"  # Dataset{gdsc or ccle}
        self.lr = 0.001  # the learning rate
        self.wd = 1e-5  # the weight decay for l2 normalizaton
        self.layer_size = [1024, 1024]  # Output sizes of every layer
        self.alpha = 0.25  # the scale for balance gcn and ni
        self.gamma = 8  # the scale for sigmod
        self.epochs = 1000  # the epochs for model


args = Args()

In [22]:
import glob
import re

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem


def load_data(args):
    """Load data based on the specified dataset."""
    if args.data == "gdsc1":
        print("load gdsc1")
        PATH = "gdsc1_data/"
        return _load_data(PATH)
    elif args.data == "gdsc2":
        print("load gdsc2")
        PATH = "gdsc2_data/"
        return _load_data(PATH)
    elif args.data == "ctrp":
        PATH = "ctrp_data/"
        return _load_data(PATH, is_ctrp=True)
    elif args.data == "nci":
        print("load nci")
        PATH = "nci_data/"
        return _load_nci(PATH)
    else:
        raise NotImplementedError


def _get_base_data(PATH):
    """Load and prepare base data common to all datasets."""
    # Load original drug response data
    drugAct = pd.read_csv(PATH + "drugAct.csv", index_col=0)

    # Load and concatenate gene expression data
    exprs = pd.concat(
        [
            pd.read_csv(PATH + "gene_exp_part1.csv.gz", index_col=0),
            pd.read_csv(PATH + "gene_exp_part2.csv.gz", index_col=0),
        ]
    ).T.dropna()

    return drugAct, exprs


def _load_data(PATH, is_ctrp=False):
    data_dir = dir_path(k=1) + PATH
    # 加载细胞系-药物矩阵

    drugAct, exprs = _get_base_data(data_dir)
    cells = sorted(
        set(drugAct.columns)
        & set(exprs.index)
        & set(pd.read_csv(data_dir + "mut.csv", index_col=0).T.index)
    )

    SMILES = pd.read_csv(data_dir + "drug2smiles.csv", index_col=0)
    exprs = exprs.loc[cells]
    drugAct = drugAct.loc[sorted(SMILES.drugs), cells]
    exprs = np.array(exprs, dtype=np.float32)

    if is_ctrp:
        drugAct = drugAct.apply(lambda x: (x - np.nanmean(x)) / np.nanstd(x))

    # Convert drug activity to binary response matrix
    res = (drugAct > 0).astype(int)
    res = np.array(res, dtype=np.float32).T

    pos_num = sp.coo_matrix(res).data.shape[0]

    # 加载药物-指纹特征矩阵
    drug_feature = pd.read_csv(
        data_dir + "nih_drug_feature.csv", index_col=0, header=0
    ).loc[sorted(SMILES.drugs)]
    drug_feature = np.array(drug_feature, dtype=np.float32)

    null_mask = (drugAct.isna()).astype(int).T
    null_mask = np.array(null_mask, dtype=np.float32)
    return res, drug_feature, exprs, null_mask, pos_num


def _load_nci(PATH):
    data_dir = dir_path(k=1) + PATH
    # 加载细胞系-药物矩阵

    drugAct, exprs = _get_base_data(data_dir)
    drugAct.columns = exprs.index
    cells = sorted(
        set(drugAct.columns)
        & set(exprs.index)
        & set(pd.read_csv(data_dir + "mut.csv", index_col=0).T.index)
    )

    # Load mechanism of action (moa) data
    moa = pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)

    # Filter drugs that have SMILES information
    drugAct = drugAct[drugAct.index.isin(moa.NSC)]

    # Load drug synonyms and filter based on availability in other datasets
    tmp = pd.read_csv("../data/drugSynonym.csv")
    tmp = tmp[
        (~tmp.nci60.isna() & ~tmp.ctrp.isna())
        | (~tmp.nci60.isna() & ~tmp.gdsc1.isna())
        | (~tmp.nci60.isna() & ~tmp.gdsc2.isna())
    ]
    tmp = [int(i) for i in set(tmp["nci60"].str.split("|").explode())]

    # Select drugs not classified as 'Other' in MOA and included in other datasets
    drugAct = drugAct.loc[
        sorted(
            set(drugAct.index)
            & (set(moa[moa["MECHANISM"] != "Other"]["NSC"]) | set(tmp))
        )
    ]

    # SMILES = pd.read_csv(data_dir + "drug2smiles.csv", index_col=0)
    exprs = exprs.loc[cells]
    drugAct = drugAct.loc[:, cells]
    exprs = np.array(exprs, dtype=np.float32)

    # Convert drug activity to binary response matrix
    res = (drugAct > 0).astype(int)
    res = np.array(res, dtype=np.float32).T

    pos_num = sp.coo_matrix(res).data.shape[0]

    # 加载药物-指纹特征矩阵
    # drug_feature = pd.read_csv(data_dir + "nih_drug_feature.csv", index_col=0, header=0)
    # drug_feature = np.array(drug_feature, dtype=np.float32)

    null_mask = (drugAct.isna()).astype(int).T
    null_mask = np.array(null_mask, dtype=np.float32)
    return res, exprs, null_mask, pos_num

In [23]:
res, exprs, null_mask, pos_num = load_data(args)

load nci


In [26]:
import numpy as np
import scipy.sparse as sp
import torch


class RandomSampler(object):
    # 元の辺をサンプリング
    # サンプリング後にテストセットと訓練セットを生成
    # 処理後の訓練セットをtorch.tensor形式に変換

    def __init__(self, adj_mat_original, train_index, test_index, null_mask, seed):
        # Initialize basic attributes
        self.seed = seed
        self.set_seed()

        self.adj_mat = to_coo_matrix(adj_mat_original)
        self.train_index = train_index
        self.test_index = test_index
        self.null_mask = null_mask

        self.train_pos = self.sample(train_index)
        self.test_pos = self.sample(test_index)
        self.train_neg, self.test_neg = self.sample_negative()
        self.train_mask = mask(self.train_pos, self.train_neg, dtype=int)
        self.test_mask = mask(self.test_pos, self.test_neg, dtype=bool)
        self.train_data = to_tensor(self.train_pos)
        self.test_data = to_tensor(self.test_pos)

    def set_seed(self):
        np.random.seed(self.seed)  # NumPyのシードを設定
        torch.manual_seed(self.seed)  # PyTorchのシードを設定

    def sample(self, index):
        row = self.adj_mat.row
        col = self.adj_mat.col
        data = self.adj_mat.data
        sample_row = row[index]
        sample_col = col[index]
        sample_data = data[index]
        sample = sp.coo_matrix(
            (sample_data, (sample_row, sample_col)), shape=self.adj_mat.shape
        )
        return sample

    def sample_negative(self):
        # identityは隣接行列が二部グラフかどうかを示す
        # 二部グラフ：辺の両端の頂点が同じ頂点集合に属していないグラフ
        pos_adj_mat = self.null_mask + self.adj_mat.toarray()
        neg_adj_mat = sp.coo_matrix(np.abs(pos_adj_mat - np.array(1)))
        all_row = neg_adj_mat.row
        all_col = neg_adj_mat.col
        all_data = neg_adj_mat.data
        index = np.arange(all_data.shape[0])

        # 負のテストセットをサンプリング
        test_n = self.test_index.shape[0]
        test_neg_index = np.random.choice(index, test_n, replace=False)
        test_row = all_row[test_neg_index]
        test_col = all_col[test_neg_index]
        test_data = all_data[test_neg_index]
        test = sp.coo_matrix(
            (test_data, (test_row, test_col)), shape=self.adj_mat.shape
        )

        # 訓練セットをサンプリング
        train_neg_index = np.delete(index, test_neg_index)
        # train_n = self.train_index.shape[0]
        # train_neg_index = np.random.choice(train_neg_index, train_n, replace=False)
        train_row = all_row[train_neg_index]
        train_col = all_col[train_neg_index]
        train_data = all_data[train_neg_index]
        train = sp.coo_matrix(
            (train_data, (train_row, train_col)), shape=self.adj_mat.shape
        )
        return train, test

In [27]:
k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

for train_index, test_index in kfold.split(np.arange(pos_num)):
    Origin_sampler = RandomSampler(res, train_index, test_index, null_mask, seed=1)
    break

In [29]:
np.alltrue((Origin_sampler.train_data == sampler.train_data).numpy())

True